# 循环老化

一个 canonical 覆盖容量/SOH、实测对标、膨胀、深度诊断和产热分析。参数变体只改顶部 CONFIG；核心计算由 src workflow 执行。


## 1. 用户配置


In [1]:
# 所有工况变体只修改本单元
CONFIG = {
    "cell": "587",
    "conditions": [
        {"temperature_c": 25, "rate_p": 0.25},
        {"temperature_c": 28, "rate_p": 0.5},
    ],
    "run_mode": "smoke",  # smoke / study
    "modes": {
        "smoke": {
            "total_cycles": 1,
            "aging_t_factor": 1,
            "return_solutions": True,
            "showprogress": False,
        },
        "study": {
            "total_cycles": 13_000,
            "aging_t_factor": 50,
            "cycles_per_block": 20,
            "return_solutions": True,
            "showprogress": True,
        },
    },
    "nominal_voltage_v": 3.2,
    "charge_cutoff_v": 3.65,
    "discharge_cutoff_v": 2.5,
    "rest_minutes": 10,
    "period_minutes": 0.5,
    "output_name": "循环老化",
    # 不需要实测时设为 None。需要时只写 registry 查询，不写磁盘绝对路径。
    "experiment": None,
    # 示例：标准 lifecycle dat 直接读取
    # "experiment": {
    #     "loader": "lifecycle_dat",
    #     "query": {"cell": "587Ah", "test_type": "循环", "format": "dat", "require_unique": True},
    # },
    # 特殊原始 Excel 使用 lifecycle_excel，并提供 sheet_name；workflow 自动写 data_processed。
    "analysis": {
        "capacity_soh": True,
        "sim_exp": True,
        "enable_swelling": True,
        "deep_analysis": True,
        "heat": True,
        "filter_conditions": "25°C",
        "heat_x_axis": "cycle",  # cycle / soh
        "swelling": {
            "omega_n": 3.1e-7,
            "omega_p": 0.0,
            "k_stiffness": 1e10,
            "preload_force": 300.0,
        },
    },
}


## 2. 环境与导入


In [3]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
plt.style.use("science")
plt.rcParams["font.family"] = "Calibri, Microsoft YaHei"
import pandas as pd

SEARCH_ROOT = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents):
    if (candidate / "src" / "easy_imports.py").exists() and (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
    nested = candidate / "BatteryProject"
    if (nested / "src" / "easy_imports.py").exists() and (nested / "pyproject.toml").exists():
        PROJECT_ROOT = nested
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Cannot locate BatteryProject root from {SEARCH_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.compare import compare_swelling
from src.easy_imports import (
    BatteryPlotter,
    compare_all,
    process_sol_list_for_all_heat_components,
    process_sol_list_with_custom_extractor,
)
from src.notebook import setup_notebook
from src.plotting import plot_analysis, plot_efficiency_vs_cycle
from src.reporting import export_cycle_metrics_report
from src.workflows.cycle import CycleWorkflowSpec, run_cycle_workflow

ctx = setup_notebook(cell=CONFIG["cell"], project_root=PROJECT_ROOT)
ANALYSIS = CONFIG["analysis"]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 3. 循环老化仿真


In [ ]:
spec = CycleWorkflowSpec.from_mapping(CONFIG)
result = run_cycle_workflow(
    spec,
    project_root=ctx.project_root,
    workspace_root=ctx.workspace_root,
)
sol_list = result["analysis_solutions"]
sim_labels_list = result["analysis_labels"]
reference_params = result["reference_params"]
display(result["metrics"])
print("Run folder:", result["context"].run_dir)


## 4. 容量/SOH 导出与绘图


In [ ]:
if ANALYSIS["capacity_soh"] and sol_list:
    report_path = result["context"].artifacts_dir / "循环容量SOH.xlsx"
    cycle_metrics_df = export_cycle_metrics_report(
        sol_list,
        sim_labels_list,
        cycle_step=spec.aging_t_factor,
        output_filename=report_path,
    )
    display(cycle_metrics_df.head())
    capacity_plotter = BatteryPlotter()
    process_sol_list_with_custom_extractor(
        capacity_plotter,
        sol_list,
        sim_labels_list,
        t_factor=spec.aging_t_factor,
    )
    capacity_plotter.plot(
        target_sim="all",
        mode="capacity",
        title_left="Discharge Capacity",
        ylabel_left="Capacity (Ah)",
        title_right="State of Health",
        ylabel_right="SOH (%)",
    )
else:
    print("容量/SOH 分析已关闭，或当前 run 未保留 solution。")


## 5. 可选：Sim vs Exp 对标


In [ ]:
exp_data = result["experiment_data"]
if ANALYSIS["sim_exp"] and exp_data and sol_list:
    compare_all(
        sol_list,
        sim_labels_list,
        exp_data_list=exp_data,
        params=reference_params,
        acceleration_factor=spec.aging_t_factor,
        metrics=["retention", "efficiency", "swelling"],
        filter_conditions=ANALYSIS["filter_conditions"],
        sim_bias={"retention": 0.0, "efficiency": 0.0, "swelling": 0.0},
        exp_bias={"retention": 0.0, "efficiency": 0.0, "swelling": 0.0},
    )
else:
    print("未配置实测数据，或 Sim vs Exp 分析已关闭。")


## 6. 可选：膨胀对标


In [ ]:
has_swelling_data = any(len(item.get("max_force", [])) for item in exp_data)
if ANALYSIS["enable_swelling"] and has_swelling_data and sol_list:
    swelling = ANALYSIS["swelling"]
    compare_swelling(
        sol_list,
        sim_labels_list,
        exp_data,
        reference_params,
        acceleration_factor=spec.aging_t_factor,
        omega_n=swelling["omega_n"],
        omega_p=swelling["omega_p"],
        k_stiffness=swelling["k_stiffness"],
        preload_force=swelling["preload_force"],
        filter_conditions=ANALYSIS["filter_conditions"],
    )
else:
    print("实验数据不含膨胀/力信号，或膨胀分析已关闭。")


## 7. 深度分析


In [ ]:
if ANALYSIS["deep_analysis"]:
    for label, solution in zip(sim_labels_list, sol_list):
        print("Efficiency analysis:", label)
        plot_efficiency_vs_cycle(solution, acceleration_factor=spec.aging_t_factor)
        print("Detailed analysis:", label)
        plot_analysis(solution, spec.aging_t_factor)
else:
    print("深度分析已关闭。")


## 8. 热分解分析


In [ ]:
if ANALYSIS["heat"] and sol_list:
    heat_plotter = BatteryPlotter()
    process_sol_list_for_all_heat_components(
        heat_plotter,
        sol_list,
        sim_labels_list,
        acceleration_factor=spec.aging_t_factor,
        x_axis=ANALYSIS["heat_x_axis"],
    )
    heat_plotter.plot(
        target_sim="all",
        title_left="Charge Heat Components",
        title_right="Discharge Heat Components",
        ylabel_left="Power (W)",
        ylabel_right="Power (W)",
        mode="heat",
        xlabel=("SOH (%)" if ANALYSIS["heat_x_axis"] == "soh" else "Cycle Number"),
        invert_xaxis=(ANALYSIS["heat_x_axis"] == "soh"),
    )
else:
    print("热分解分析已关闭，或当前 run 未保留 solution。")
